In [8]:
import os
import itertools
import pandas as pd
from IPython.display import display

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN (Dựa theo ảnh thư mục của bạn)
# ==========================================
# Vì notebook đang nằm ở 11/notebooks/, ta cần lùi lại một bậc (../) để trỏ tới thư mục gốc
JULIA_SCRIPT = '../src/algorithm/relim.jl'
DATA_PATH = '../data/benchmark/retail.txt'
FILE_PATH = 'output_retail.txt' # File này sẽ được sinh ra ngay tại thư mục notebooks/

TOTAL_TRANSACTIONS = 88162 # Tổng số dòng của file retail.txt
MIN_CONFIDENCE = 0.5       # Ngưỡng tin cậy 50%

# ==========================================
# BƯỚC 1: TỰ ĐỘNG GỌI JULIA ĐỂ TẠO FILE OUTPUT
# ==========================================
print("BƯỚC 1: Đang chạy ngầm thuật toán Relim (Julia) để lấy Frequent Itemsets...")
print("   Vui lòng đợi khoảng 5-10 giây...\n")

# Lệnh ! giúp Jupyter Notebook chạy lệnh Terminal trực tiếp
# Chạy thuật toán cơ bản với minsup = 0.05 (5%)
!julia --project=.. {JULIA_SCRIPT} -i {DATA_PATH} -o {FILE_PATH} -m 0.05 -a basic

print("\nĐã tạo thành công file kết quả!")

# ==========================================
# BƯỚC 2: HÀM SINH LUẬT KẾT HỢP
# ==========================================
def load_frequent_itemsets(filepath):
    itemsets = {}
    with open(filepath, 'r') as f:
        for line in f:
            if not line.strip(): continue
            parts = line.strip().split('#SUP:')
            items = tuple(sorted([int(x) for x in parts[0].strip().split()]))
            sup_count = int(parts[1].strip())
            itemsets[items] = sup_count
    return itemsets

def generate_association_rules(itemsets, total_transactions, min_conf):
    rules = []
    for itemset, sup_Z in itemsets.items():
        n = len(itemset)
        if n < 2: continue
            
        for i in range(1, n):
            for subset_X in itertools.combinations(itemset, i):
                X = tuple(sorted(subset_X))
                Y = tuple(sorted(set(itemset) - set(X)))
                
                sup_X = itemsets.get(X, 0)
                sup_Y = itemsets.get(Y, 0)
                if sup_X == 0 or sup_Y == 0: continue
                
                conf = sup_Z / sup_X
                if conf >= min_conf:
                    prob_Z = sup_Z / total_transactions
                    prob_X = sup_X / total_transactions
                    prob_Y = sup_Y / total_transactions
                    
                    lift = prob_Z / (prob_X * prob_Y)
                    
                    rules.append({
                        'Vế Trái (X)': str(list(X)),
                        'Vế Phải (Y)': str(list(Y)),
                        'Support (X U Y)': sup_Z,
                        'Confidence': round(conf, 4),
                        'Lift': round(lift, 4)
                    })
    return rules

# ==========================================
# BƯỚC 3: THỰC THI VÀ XUẤT BẢNG TOP 10
# ==========================================
print("\n⏳ BƯỚC 2: Đang sinh Luật kết hợp (Association Rules)...")
frequent_itemsets = load_frequent_itemsets(FILE_PATH)
print(f"-> Đã tải {len(frequent_itemsets)} tập phổ biến.")

rules_list = generate_association_rules(frequent_itemsets, TOTAL_TRANSACTIONS, MIN_CONFIDENCE)

df_rules = pd.DataFrame(rules_list)

if not df_rules.empty:
    top_10_rules = df_rules.sort_values(by='Lift', ascending=False).head(10).reset_index(drop=True)
    top_10_rules.index += 1
    
    print("\nTOP 10 LUẬT KẾT HỢP MẠNH NHẤT (Sắp xếp theo hệ số Lift):")
    display(top_10_rules)
else:
    print("Không tìm thấy luật nào. Hãy thử giảm minsup lúc chạy thuật toán xuống thấp hơn (vd: -m 0.005).")

BƯỚC 1: Đang chạy ngầm thuật toán Relim (Julia) để lấy Frequent Itemsets...
   Vui lòng đợi khoảng 5-10 giây...

>>> Sử dụng phiên bản: Relim BASIC
>>> Đang nạp dataset: ../data/benchmark/retail.txt
Đã tải 88162 giao dịch.
Min Support Tuyệt Đối: 4409
>>> Bắt đầu chạy Đệ quy...
Khai thác hoàn tất! Tìm thấy 16 tập phổ biến.
Thời gian chạy: 0.3945 giây.
>>> Đang xuất file vào: output_retail.txt
Mọi quy trình hoàn thành!

Đã tạo thành công file kết quả!

⏳ BƯỚC 2: Đang sinh Luật kết hợp (Association Rules)...
-> Đã tải 16 tập phổ biến.

TOP 10 LUẬT KẾT HỢP MẠNH NHẤT (Sắp xếp theo hệ số Lift):


,Vế Trái (X),Vế Phải (Y),Support (X U Y),Confidence,Lift
1,"[41, 48]",[39],7366,0.8168,1.4210
2,"[39, 41]",[48],7366,0.6453,1.3503
3,"[32, 39]",[48],5402,0.6389,1.3368
4,"[38, 48]",[39],6102,0.7681,1.3364
5,[41],[39],11414,0.7637,1.3287
6,[41],[48],9018,0.6034,1.2626
7,"[38, 39]",[48],6102,0.5899,1.2342
8,[48],[39],29142,0.6916,1.2033
9,[39],[48],29142,0.5751,1.2033
10,"[32, 48]",[39],5402,0.6724,1.1698
